# Fitting the CMA Model to Kaggle Brist1D CGM Data

In [ ]:
# Install the Kaggle CLI if you haven't already
!pip install kaggle

# Ensure your kaggle.json is correctly configured before running this command
!kaggle competitions download -c brist1d
!unzip -o brist1d.zip

import pandas as pd
import numpy as np

# Load CGM data from the downloaded Brist1D dataset
df = pd.read_csv('train.csv')

# For this example, we will focus on a single participant's continuous glucose trace
participant_df = df[df['p_num'] == 1].copy().reset_index(drop=True)

# The model requires 'time' in UTC format, and typically 'G' (glucose) and 'sg' (sensor glucose) columns
# In the Kaggle dataset, 'time' is just HH:MM:SS, and 'bg-0:00' is the current blood glucose reading.
# We synthesize full dates starting from today for demonstration purposes.
participant_df['time'] = pd.to_datetime('2023-01-01 ' + participant_df['time']).dt.tz_localize('UTC')
participant_df['G'] = participant_df['bg-0:00']
participant_df['sg'] = participant_df['bg-0:00']

# Drop NaNs to ensure the model fits cleanly
cgm_data = participant_df[['time', 'G', 'sg']].dropna().head(100) # taking the first 100 continuous points

# Checking the structure of the loaded data
print(cgm_data.head())


In [ ]:
from pfun_cma_model.engine.fit import fit_model

# Fit the model to the CGM data
model_results = fit_model(cgm_data, tcol='time', ycol='G')

print("Model Parameters:")
print(model_results.popt_named)
print("\nModel Fit Results Info:")
print(model_results.infodict['message'])


In [ ]:
# Displaying results
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
plt.plot(cgm_data['time'], cgm_data['G'], label='CGM Data', marker='o', linestyle='none', alpha=0.5)

# The model_results.soln dataframe contains the fitted curve
plt.plot(model_results.formatted_data['time'], model_results.soln['G'], label='Fitted CMA Model', color='red', linewidth=2)

plt.title('Blood Glucose Levels over Time (CMA Model Fit)')
plt.xlabel('Time')
plt.ylabel('Blood Glucose')
plt.legend()
plt.show()


## Posthoc Analysis

The PFun CMA model effectively characterizes the dynamic continuous glucose monitor (CGM) traces by adjusting specific physiological parameters (`taup`, `taug`, `Cm`).

By applying the model directly to the Kaggle Brist1D dataset, we observe that the CMA model is capable of fitting its compartmental equations to the raw glucose (`G`) values, providing the solid 'red' regression line above. The `infodict` generated through the fitting process yields vital metrics, like termination messages, to confirm the algorithm successfully converged on optimized coefficients.

**Key Highlights:**
- The fitting process expects timestamp columns (`time` by default) in **UTC** format, which we handled by appending arbitrary UTC dates to the `HH:MM:SS` time values provided in the competition dataset.
- Additional preprocessing steps are often handled directly through `fit_model` calling `format_data`, which necessitates the presence of both `G` and `sg` columns in typical evaluation datasets. We map `bg-0:00` to both here.
- By observing the parameter bounds and bounds conditions, one can tune the CMA model further to adjust to real physiological profiles seen across different participants.